# Model-Agnostic Axis 2 Artifact Builder

This notebook builds the full artifact set needed by the summary notebook from a single best checkpoint.

Outputs are written to a model-specific output folder and include:
1. per_bit_auroc/*
2. threshold_sweep/*
3. retrieval/*
4. base caches (y_pred/y_true for val and OOD)

Recommended chain:
1. Run this notebook per model checkpoint (6 runs).
2. Run axis_2_analysis_summary notebook per model output folder.
3. Run cross-axis join notebook that combines per-bit AUROC with correlation prep outputs.

In [2]:
import os
import sys
import json
import shutil
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity

from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys

try:
    from map4 import MAP4
    MAP4_AVAILABLE = True
except Exception:
    MAP4_AVAILABLE = False

PROJECT_ROOT = Path(os.getcwd()).parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dreams.models.heads.heads import FingerprintHead
from dreams.definitions import PRETRAINED

DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

/Users/wouterachterberg/coding/DreaMS/.venv/lib/python3.10/site-packages/lightning_fabric/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
/Users/wouterachterberg/coding/DreaMS/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/User

DEVICE: mps


In [49]:
# ------------------------------------------------------------------
# Config
# ------------------------------------------------------------------
# External-drive checkpoint root (override if your mount point changes)
CKPT_BASE_DIR = Path('/Volumes/NVMe_Wouter/THESIS/model_checkpoints')

# Fill this once, then run by changing RUN_INDEX only.
MODEL_SPECS = [
    {
        'run_tag': 'morgan_2048_cos',
        'fp_kind': 'morgan_2048',
        'loss_kind': 'cos',
        'model_type': 'fingerprint_head',
        'ckpt_file': 'epoch=63-step=8382-val_loss=0.361643.ckpt',
        'apply_sigmoid_to_pred': False,
    },
    {
        'run_tag': 'morgan_2048_bce',
        'fp_kind': 'morgan_2048',
        'loss_kind': 'bce',
        'model_type': 'fingerprint_head',
        'ckpt_file': 'epoch=31-step=4224-val_loss=0.061447.ckpt',
        'apply_sigmoid_to_pred': True,
    },
    {
        'run_tag': 'maccs_166_cos',
        'fp_kind': 'maccs_166',
        'loss_kind': 'cos',
        'model_type': 'fingerprint_head',
        'ckpt_file': 'epoch=19-step=2640-val_loss=0.135409.ckpt',
        'apply_sigmoid_to_pred': False,
    },
    {
        'run_tag': 'maccs_166_bce',
        'fp_kind': 'maccs_166',
        'loss_kind': 'bce',
        'model_type': 'fingerprint_head',
        'ckpt_file': 'epoch=11-step=1584-val_loss=0.237082.ckpt',
        'apply_sigmoid_to_pred': True,
    },
    {
        'run_tag': 'map4_2048_cos',
        'fp_kind': 'map4_2048',
        'loss_kind': 'cos',
        'model_type': 'fingerprint_head',
        'ckpt_file': 'epoch=23-step=3168-val_loss=0.420152.ckpt',
        'apply_sigmoid_to_pred': False,
    },
    {
        'run_tag': 'map4_2048_bce',
        'fp_kind': 'map4_2048',
        'loss_kind': 'bce',
        'model_type': 'fingerprint_head',
        'ckpt_file': 'epoch=17-step=2310-val_loss=0.453639.ckpt',
        'apply_sigmoid_to_pred': True,
    },
    # Frozen baselines from results/frozen_deepsets_baselines
    {
        'run_tag': 'morgan_2048_bce_frozen',
        'fp_kind': 'morgan_2048',
        'loss_kind': 'bce',
        'model_type': 'frozen_head',
        'ckpt_path': str(PROJECT_ROOT / 'dreams-thesis-wa/results/frozen_deepsets_baselines/frozen_morgan_2048_bce/checkpoints/frozen_morgan_2048_bce_best.ckpt'),
        'apply_sigmoid_to_pred': True,
    },
    {
        'run_tag': 'maccs_166_bce_frozen',
        'fp_kind': 'maccs_166',
        'loss_kind': 'bce',
        'model_type': 'frozen_head',
        'ckpt_path': str(PROJECT_ROOT / 'dreams-thesis-wa/results/frozen_deepsets_baselines/frozen_maccs_166_bce/checkpoints/frozen_maccs_166_bce_best.ckpt'),
        'apply_sigmoid_to_pred': True,
    },
    {
        'run_tag': 'map4_2048_bce_frozen',
        'fp_kind': 'map4_2048',
        'loss_kind': 'bce',
        'model_type': 'frozen_head',
        'ckpt_path': str(PROJECT_ROOT / 'dreams-thesis-wa/results/frozen_deepsets_baselines/frozen_map4_2048_bce/checkpoints/frozen_map4_2048_bce_best.ckpt'),
        'apply_sigmoid_to_pred': True,
    },
    {
        'run_tag': 'morgan_2048_cos_frozen',
        'fp_kind': 'morgan_2048',
        'loss_kind': 'cos',
        'model_type': 'frozen_head',
        'ckpt_path': str(PROJECT_ROOT / 'dreams-thesis-wa/results/frozen_deepsets_baselines/frozen_morgan_2048_cos/checkpoints/frozen_morgan_2048_cos_best.ckpt'),
        'apply_sigmoid_to_pred': False,
    },
    {
        'run_tag': 'maccs_166_cos_frozen',
        'fp_kind': 'maccs_166',
        'loss_kind': 'cos',
        'model_type': 'frozen_head',
        'ckpt_path': str(PROJECT_ROOT / 'dreams-thesis-wa/results/frozen_deepsets_baselines/frozen_maccs_166_cos/checkpoints/frozen_maccs_166_cos_best.ckpt'),
        'apply_sigmoid_to_pred': False,
    },
    {
        'run_tag': 'map4_2048_cos_frozen',
        'fp_kind': 'map4_2048',
        'loss_kind': 'cos',
        'model_type': 'frozen_head',
        'ckpt_path': str(PROJECT_ROOT / 'dreams-thesis-wa/results/frozen_deepsets_baselines/frozen_map4_2048_cos/checkpoints/frozen_map4_2048_cos_best.ckpt'),
        'apply_sigmoid_to_pred': False,
    },
]

# Change only this value per run, or pass RUN_INDEX via env.
# 0-5: original fine-tuned runs, 6-11: frozen runs.
RUN_INDEX = int(os.environ.get('RUN_INDEX', '5'))
REUSE_CACHED_PRED = os.environ.get('REUSE_CACHED_PRED', '1') != '0'

if RUN_INDEX < 0 or RUN_INDEX >= len(MODEL_SPECS):
    raise IndexError(f'RUN_INDEX={RUN_INDEX} out of range for {len(MODEL_SPECS)} model specs.')

spec = MODEL_SPECS[RUN_INDEX]
FP_KIND = spec['fp_kind']          # one of: morgan_2048, maccs_166, map4_2048
LOSS_KIND = spec['loss_kind']      # one of: cos, bce
MODEL_TYPE = spec.get('model_type', 'fingerprint_head')
RUN_TAG = spec.get('run_tag', f'{FP_KIND}_{LOSS_KIND}')
APPLY_SIGMOID_TO_PRED = bool(spec.get('apply_sigmoid_to_pred', LOSS_KIND == 'bce'))

ckpt_file = spec.get('ckpt_file')
ckpt_path = spec.get('ckpt_path')
if ckpt_path:
    CKPT_PATH = Path(ckpt_path)
elif ckpt_file:
    if 'XX' in ckpt_file:
        raise ValueError(
            f"Placeholder checkpoint filename detected for {RUN_TAG}: {ckpt_file}\n"
            "Replace ckpt_file with a real filename from CKPT_BASE_DIR."
        )
    CKPT_PATH = CKPT_BASE_DIR / ckpt_file
else:
    raise ValueError(f"Model spec for {RUN_TAG} needs either 'ckpt_file' or 'ckpt_path'.")

PROBING_TEST_PATH = PROJECT_ROOT / 'dreams-thesis-wa/data/processed/MassSpecGym_splits/probing_test.parquet'
FINETUNING_HDF5_PATH = PROJECT_ROOT / 'dreams-thesis-wa/data/processed/MassSpecGym_splits/finetuning.hdf5'
FINETUNING_WITH_SSL_HDF5_PATH = PROJECT_ROOT / 'dreams-thesis-wa/data/processed/MassSpecGym_splits/finetuning_with_ssl_embeddings.hdf5'

# Canonical output convention:
# dreams-thesis-wa/results/model_runs/<run_tag>/
#   checkpoints/<original_checkpoint_filename>.ckpt
#   axis2_artifacts/
#     per_bit_auroc/
#     threshold_sweep/
#     retrieval/
BASE_OUTPUT = PROJECT_ROOT / 'dreams-thesis-wa/results/model_runs'
MODEL_RUN_DIR = BASE_OUTPUT / RUN_TAG
CHECKPOINTS_DIR = MODEL_RUN_DIR / 'checkpoints'
MODEL_OUTPUT_DIR = MODEL_RUN_DIR / 'axis2_artifacts'
AUROC_DIR = MODEL_OUTPUT_DIR / 'per_bit_auroc'
SWEEP_DIR = MODEL_OUTPUT_DIR / 'threshold_sweep'
RETRIEVAL_DIR = MODEL_OUTPUT_DIR / 'retrieval'
CACHE_Y_PRED_PATH = MODEL_OUTPUT_DIR / 'y_pred.npy'
CACHE_Y_TRUE_PATH = MODEL_OUTPUT_DIR / 'y_true.npy'
CACHE_Y_PRED_VAL_PATH = MODEL_OUTPUT_DIR / 'y_pred_val.npy'
CACHE_Y_TRUE_VAL_PATH = MODEL_OUTPUT_DIR / 'y_true_val.npy'

MODEL_RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUROC_DIR.mkdir(parents=True, exist_ok=True)
SWEEP_DIR.mkdir(parents=True, exist_ok=True)
RETRIEVAL_DIR.mkdir(parents=True, exist_ok=True)

USE_CACHED_PREDICTIONS = (
    REUSE_CACHED_PRED
    and MODEL_TYPE == 'fingerprint_head'
    and CACHE_Y_PRED_PATH.exists()
    and CACHE_Y_TRUE_PATH.exists()
    and CACHE_Y_PRED_VAL_PATH.exists()
    and CACHE_Y_TRUE_VAL_PATH.exists()
)

print('Selected model spec index:', RUN_INDEX)
print('Checkpoint base dir:', CKPT_BASE_DIR)
print('Checkpoint input:', CKPT_PATH)
print('FP kind:', FP_KIND)
print('Loss kind:', LOSS_KIND)
print('Model type:', MODEL_TYPE)
print('Apply sigmoid:', APPLY_SIGMOID_TO_PRED)
print('Reuse cached predictions:', REUSE_CACHED_PRED)
print('Using cached predictions for this run:', USE_CACHED_PREDICTIONS)
print('Run tag:', RUN_TAG)
print('Model run dir:', MODEL_RUN_DIR)
print('Artifacts output dir:', MODEL_OUTPUT_DIR)

if not CKPT_PATH.exists():
    raise FileNotFoundError(f'Checkpoint not found: {CKPT_PATH}')
if FP_KIND.startswith('map4_') and not MAP4_AVAILABLE:
    raise RuntimeError('MAP4 package not available in current environment.')
if MODEL_TYPE == 'frozen_head' and not FINETUNING_WITH_SSL_HDF5_PATH.exists():
    raise FileNotFoundError(f'Frozen runs need embeddings file: {FINETUNING_WITH_SSL_HDF5_PATH}')

# Keep a reproducible checkpoint copy in each run folder using its original filename.
RUN_CKPT_PATH = CHECKPOINTS_DIR / CKPT_PATH.name
if CKPT_PATH.resolve() != RUN_CKPT_PATH.resolve():
    shutil.copy2(CKPT_PATH, RUN_CKPT_PATH)
    print('Copied checkpoint to:', RUN_CKPT_PATH)
else:
    print('Using checkpoint path:', RUN_CKPT_PATH)

# Use run-local checkpoint path from now on for full reproducibility.
CKPT_PATH = RUN_CKPT_PATH

Selected model spec index: 5
Checkpoint base dir: /Volumes/NVMe_Wouter/THESIS/model_checkpoints
Checkpoint input: /Volumes/NVMe_Wouter/THESIS/model_checkpoints/epoch=17-step=2310-val_loss=0.453639.ckpt
FP kind: map4_2048
Loss kind: bce
Model type: fingerprint_head
Apply sigmoid: True
Reuse cached predictions: True
Using cached predictions for this run: True
Run tag: map4_2048_bce
Model run dir: /Users/wouterachterberg/coding/DreaMS/dreams-thesis-wa/results/model_runs/map4_2048_bce
Artifacts output dir: /Users/wouterachterberg/coding/DreaMS/dreams-thesis-wa/results/model_runs/map4_2048_bce/axis2_artifacts
Copied checkpoint to: /Users/wouterachterberg/coding/DreaMS/dreams-thesis-wa/results/model_runs/map4_2048_bce/checkpoints/epoch=17-step=2310-val_loss=0.453639.ckpt


In [50]:
# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------
class FrozenEmbeddingDeepSetsHead(nn.Module):
    """Simple DeepSets-style head used by frozen checkpoints."""

    def __init__(self, out_dim: int, dropout: float = 0.0):
        super().__init__()
        self.phi = nn.Sequential(
            nn.Linear(1024, 1024, bias=False),
            nn.Dropout(dropout),
        )
        self.rho = nn.Linear(1024, out_dim, bias=False)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        # Supports both [B, 1024] (single vector) and [B, P, 1024] (peak set).
        if x.ndim == 2:
            x = x.unsqueeze(1)  # [B, 1024] -> [B, 1, 1024]

        x = self.phi(x)
        if mask is not None:
            # mask expected as [B, P], values in {0,1}.
            m = mask.float().unsqueeze(-1)
            x = x * m
        x = x.sum(dim=1)
        return self.rho(x)


def parse_spectrum_strings(mzs_str, intens_str, n_peaks=128):
    mzs = np.fromstring(str(mzs_str), sep=',', dtype=np.float32)
    ints = np.fromstring(str(intens_str), sep=',', dtype=np.float32)
    if len(mzs) == 0 or len(ints) == 0:
        return np.zeros((2, n_peaks), dtype=np.float32)
    n = min(len(mzs), len(ints))
    mzs, ints = mzs[:n], ints[:n]
    order = np.argsort(ints)[::-1][:n_peaks]
    mzs, ints = mzs[order], ints[order]
    order_mz = np.argsort(mzs)
    mzs, ints = mzs[order_mz], ints[order_mz]
    if ints.max() > 0:
        ints = ints / ints.max()
    out = np.zeros((2, n_peaks), dtype=np.float32)
    out[0, :len(mzs)] = mzs
    out[1, :len(ints)] = ints
    return out


_map4_cache = {}
def _map4_calc_for_dim(dim):
    if dim not in _map4_cache:
        _map4_cache[dim] = MAP4(dimensions=dim, radius=2)
    return _map4_cache[dim]


def fp_from_smiles(smiles, fp_kind):
    mol = Chem.MolFromSmiles(smiles)

    # Determine target length from fp_kind.
    if fp_kind == 'morgan_2048':
        n_bits = 2048
    elif fp_kind == 'maccs_166':
        n_bits = 166
    elif fp_kind == 'map4_1024':
        n_bits = 1024
    elif fp_kind == 'map4_2048':
        n_bits = 2048
    else:
        raise ValueError(f'Unsupported fp kind: {fp_kind}')

    if mol is None:
        return np.zeros((n_bits,), dtype=np.float32)

    if fp_kind == 'morgan_2048':
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
        return np.array(fp, dtype=np.float32)
    if fp_kind == 'maccs_166':
        fp = MACCSkeys.GenMACCSKeys(mol)
        return np.array(fp, dtype=np.float32)[1:]
    if fp_kind == 'map4_1024':
        calc = _map4_calc_for_dim(1024)
        x = np.asarray(calc.calculate(mol), dtype=np.float32)
        return (x != 0).astype(np.float32)
    if fp_kind == 'map4_2048':
        calc = _map4_calc_for_dim(2048)
        x = np.asarray(calc.calculate(mol), dtype=np.float32)
        return (x != 0).astype(np.float32)

    raise ValueError(f'Unsupported fp kind: {fp_kind}')


def _ensure_model_spec_layout(spectra_np):
    """
    Normalize spectra layout to [N, n_peaks, 2] (mz, intensity),
    which is what FingerprintHead expects internally after preprocessing.
    """
    arr = np.asarray(spectra_np, dtype=np.float32)
    if arr.ndim != 3:
        raise ValueError(f'Expected spectra with 3 dims [N,*,*], got shape={arr.shape}')

    # Already [N, n_peaks, 2]
    if arr.shape[-1] == 2:
        return arr

    # Common alternate layout from preprocessing: [N, 2, n_peaks]
    if arr.shape[1] == 2:
        return np.transpose(arr, (0, 2, 1)).copy()

    raise ValueError(
        f'Unsupported spectra layout {arr.shape}. Expected [N,n_peaks,2] or [N,2,n_peaks].'
    )


def infer_batches(model, spectra_np, prec_mz_np=None, batch_size=256):
    spectra_np = _ensure_model_spec_layout(spectra_np)

    # Reuse the exact preprocessor that was used to train the loaded backbone.
    spec_preproc = getattr(getattr(model, 'backbone', None), 'spec_preproc', None)
    if spec_preproc is None:
        raise RuntimeError('Loaded model has no backbone.spec_preproc; cannot reproduce training-time preprocessing.')

    prec_arr = None
    if prec_mz_np is not None:
        prec_arr = np.asarray(prec_mz_np, dtype=np.float32)
        if len(prec_arr) != len(spectra_np):
            raise ValueError(f'prec_mz length mismatch: spectra={len(spectra_np)} prec={len(prec_arr)}')

    n = len(spectra_np)
    out = []
    with torch.no_grad():
        for start in tqdm(range(0, n, batch_size), desc='Inference'):
            end = min(start + batch_size, n)
            batch_spec_raw = spectra_np[start:end]

            # Apply SpectrumPreprocessor so eval matches training input format.
            batch_spec_proc = []
            for i, spec in enumerate(batch_spec_raw):
                prec = None
                if prec_arr is not None:
                    p = float(prec_arr[start + i])
                    if np.isfinite(p):
                        prec = p
                batch_spec_proc.append(spec_preproc(spec, prec_mz=prec, high_form='auto', augment=False))

            batch_spec = torch.tensor(np.asarray(batch_spec_proc, dtype=np.float32), dtype=torch.float32, device=DEVICE)
            batch_charge = torch.ones(end - start, dtype=torch.float32, device=DEVICE)
            pred = model(batch_spec, batch_charge)
            if APPLY_SIGMOID_TO_PRED:
                pred = torch.sigmoid(pred)
            out.append(pred.detach().cpu().numpy().astype(np.float32))
    return np.concatenate(out, axis=0)


def infer_frozen_batches(model, embeddings_np, embedding_masks_np=None, batch_size=256):
    embs = np.asarray(embeddings_np, dtype=np.float32)
    if embs.ndim == 2 and embs.shape[1] != 1024:
        raise ValueError(f'Expected embeddings [N,1024] for 2D input, got {embs.shape}')
    if embs.ndim == 3 and embs.shape[2] != 1024:
        raise ValueError(f'Expected embeddings [N,P,1024] for 3D input, got {embs.shape}')
    if embs.ndim not in {2, 3}:
        raise ValueError(f'Expected 2D or 3D embeddings, got {embs.shape}')

    masks = None
    if embedding_masks_np is not None:
        masks = np.asarray(embedding_masks_np, dtype=np.float32)
        if masks.ndim != 2:
            raise ValueError(f'Expected mask shape [N,P], got {masks.shape}')
        if embs.ndim == 3 and masks.shape[1] != embs.shape[1]:
            raise ValueError(f'Mask/embedding peak mismatch: mask={masks.shape}, emb={embs.shape}')
        if masks.shape[0] != embs.shape[0]:
            raise ValueError(f'Mask/embedding batch mismatch: mask={masks.shape}, emb={embs.shape}')

    n = len(embs)
    out = []
    with torch.no_grad():
        for start in tqdm(range(0, n, batch_size), desc='Frozen inference'):
            end = min(start + batch_size, n)
            xb = torch.tensor(embs[start:end], dtype=torch.float32, device=DEVICE)
            mb = None if masks is None else torch.tensor(masks[start:end], dtype=torch.float32, device=DEVICE)

            # Backward compatible call for old frozen heads that ignore mask.
            if mb is None:
                pred = model(xb)
            else:
                pred = model(xb, mb)

            if APPLY_SIGMOID_TO_PRED:
                pred = torch.sigmoid(pred)
            out.append(pred.detach().cpu().numpy().astype(np.float32))
    return np.concatenate(out, axis=0)


def compute_per_bit_auroc(y_true, y_pred):
    rows = []
    for b in range(y_true.shape[1]):
        yt = y_true[:, b]
        yp = y_pred[:, b]
        freq = float(yt.mean())
        if yt.min() == yt.max():
            rows.append((b, np.nan, freq))
        else:
            rows.append((b, float(roc_auc_score(yt, yp)), freq))
    return pd.DataFrame(rows, columns=['bit_index', 'auroc', 'freq'])


def compute_rowwise_cosine_stats(y_true, y_pred):
    numer = np.sum(y_true * y_pred, axis=1)
    denom = np.linalg.norm(y_true, axis=1) * np.linalg.norm(y_pred, axis=1)
    sims = numer / np.maximum(denom, 1e-8)
    return {
        'cosine_sim_mean': float(np.mean(sims)),
        'cosine_sim_median': float(np.median(sims)),
    }


def sweep_thresholds(y_pred, y_true, thresholds):
    rows = []
    for t in thresholds:
        yb = (y_pred >= t).astype(np.uint8)
        tp = (yb & y_true.astype(np.uint8)).sum(axis=1)
        fp = (yb & (1 - y_true.astype(np.uint8))).sum(axis=1)
        fn = ((1 - yb) & y_true.astype(np.uint8)).sum(axis=1)
        union = ((yb | y_true.astype(np.uint8))).sum(axis=1)

        precision = np.where(tp + fp > 0, tp / (tp + fp), 0.0).mean()
        recall = np.where(tp + fn > 0, tp / (tp + fn), 0.0).mean()
        f1 = np.where(precision + recall > 0, 2 * precision * recall / (precision + recall), 0.0)
        tanimoto = np.where(union > 0, tp / union, 0.0).mean()
        rows.append((float(t), float(precision), float(recall), float(f1), float(tanimoto)))
    return pd.DataFrame(rows, columns=['threshold', 'precision', 'recall', 'f1', 'tanimoto_mean'])

In [51]:
# ------------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------------
import pathlib


def _load_torch_checkpoint_compat(ckpt_path):
    """Load checkpoint with torch>=2.6 compatibility."""
    try:
        torch.serialization.add_safe_globals([pathlib.PosixPath])
    except Exception:
        pass

    original_torch_load = torch.load

    def _torch_load_compat(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return original_torch_load(*args, **kwargs)

    torch.load = _torch_load_compat
    try:
        ckpt = original_torch_load(str(ckpt_path), map_location=torch.device('cpu'), weights_only=False)
    finally:
        torch.load = original_torch_load

    return ckpt


def _load_fingerprint_head_checkpoint(ckpt_path):
    """Load Lightning FingerprintHead checkpoints (fine-tuned runs)."""
    try:
        torch.serialization.add_safe_globals([pathlib.PosixPath])
    except Exception:
        pass

    original_torch_load = torch.load

    def _torch_load_compat(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return original_torch_load(*args, **kwargs)

    torch.load = _torch_load_compat
    try:
        m = FingerprintHead.load_from_checkpoint(
            str(ckpt_path),
            backbone=PRETRAINED / 'ssl_model.ckpt',
            map_location=torch.device('cpu')
        )
    finally:
        torch.load = original_torch_load

    return m.eval().float().to(DEVICE)


def _load_frozen_head_checkpoint(ckpt_path):
    """Load plain torch checkpoints produced by frozen baseline notebook."""
    ckpt = _load_torch_checkpoint_compat(ckpt_path)
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        state = ckpt['model_state_dict']
    elif isinstance(ckpt, dict):
        state = ckpt
    else:
        raise TypeError(f'Unexpected frozen checkpoint object type: {type(ckpt)}')

    if 'rho.weight' not in state:
        raise KeyError("Frozen checkpoint missing 'rho.weight'; cannot infer output size.")

    out_dim = int(state['rho.weight'].shape[0])
    m = FrozenEmbeddingDeepSetsHead(out_dim=out_dim, dropout=0.0)
    missing, unexpected = m.load_state_dict(state, strict=False)
    if missing:
        print('Warning: missing keys while loading frozen checkpoint:', missing)
    if unexpected:
        print('Warning: unexpected keys while loading frozen checkpoint:', unexpected)

    m = m.eval().float().to(DEVICE)
    return m, ckpt


if USE_CACHED_PREDICTIONS:
    model = None
    print('Skipping checkpoint load because cached predictions are available.')
elif MODEL_TYPE == 'frozen_head':
    model, frozen_ckpt_meta = _load_frozen_head_checkpoint(CKPT_PATH)
    print('Loaded frozen checkpoint:', CKPT_PATH.name)
    print('Frozen out_dim:', int(model.rho.weight.shape[0]))
    if isinstance(frozen_ckpt_meta, dict):
        if 'epoch' in frozen_ckpt_meta:
            print('Frozen checkpoint epoch:', frozen_ckpt_meta['epoch'])
        if 'val_loss' in frozen_ckpt_meta:
            print('Frozen checkpoint val_loss:', frozen_ckpt_meta['val_loss'])
else:
    model = _load_fingerprint_head_checkpoint(CKPT_PATH)
    print('Loaded checkpoint:', CKPT_PATH.name)
    print('Model fp_str:', getattr(model, 'fp_str', 'unknown'))
    print('Model fp_size:', getattr(model, 'fp_size', 'unknown'))

Skipping checkpoint load because cached predictions are available.


In [52]:
# ------------------------------------------------------------------
# Build OOD (probing_test) and validation sets
# ------------------------------------------------------------------
df_ood = pd.read_parquet(PROBING_TEST_PATH)
smiles_ood = df_ood['smiles'].astype(str).tolist()

if MODEL_TYPE == 'frozen_head':
    emb_mask_ood = None
    emb_mask_val = None
    prec_ood = None
    prec_val = None

    # OOD embeddings from parquet.
    emb_col = 'ssl_embedding' if 'ssl_embedding' in df_ood.columns else ('embedding' if 'embedding' in df_ood.columns else None)
    if emb_col is None:
        raise KeyError('Frozen run requires ssl_embedding/embedding column in probing_test.parquet')
    emb_ood = np.asarray(list(df_ood[emb_col].values), dtype=np.float32)

    # Optional OOD mask columns for all-peaks frozen runs.
    for mk in ['ssl_peak_mask', 'peak_mask', 'embedding_mask']:
        if mk in df_ood.columns:
            emb_mask_ood = np.asarray(list(df_ood[mk].values), dtype=np.float32)
            break

    with h5py.File(FINETUNING_WITH_SSL_HDF5_PATH, 'r') as f:
        fold = np.array([x.decode('utf-8') if isinstance(x, bytes) else str(x) for x in f['fold'][:]])
        smiles_all = np.array([x.decode('utf-8') if isinstance(x, bytes) else str(x) for x in f['smiles'][:]])

        emb_key = None
        # Prefer all-peaks embeddings when available for future frozen-allpeaks runs.
        for k in ['ssl_peak_embeddings', 'peak_embeddings', 'ssl_embedding', 'embedding', 'ssl_embeddings', 'embeddings']:
            if k not in f:
                continue
            shp = f[k].shape
            if len(shp) == 3 and shp[2] == 1024:
                emb_key = k
                break
            if len(shp) == 2 and shp[1] == 1024:
                emb_key = k
                break
        if emb_key is None:
            raise KeyError(f'No embedding dataset found in {FINETUNING_WITH_SSL_HDF5_PATH}. Keys={list(f.keys())}')
        emb_all = f[emb_key][:].astype(np.float32)

        # Optional mask from HDF5 for all-peaks embeddings.
        if emb_all.ndim == 3 and 'ssl_peak_mask' in f:
            emb_mask_all = f['ssl_peak_mask'][:].astype(np.float32)
        else:
            emb_mask_all = None

    val_mask = fold == 'val'
    if val_mask.sum() == 0:
        val_mask = fold == 'test'

    smiles_val = smiles_all[val_mask].tolist()
    emb_val = emb_all[val_mask]
    if emb_mask_all is not None:
        emb_mask_val = emb_mask_all[val_mask]

    print(f'Using OOD embedding column: {emb_col}')
    print(f'Using finetuning embedding key: {emb_key}')
    print('OOD embeddings:', len(smiles_ood), emb_ood.shape)
    if emb_mask_ood is not None:
        print('OOD masks:', emb_mask_ood.shape)
    print('VAL embeddings:', len(smiles_val), emb_val.shape)
    if emb_mask_val is not None:
        print('VAL masks:', emb_mask_val.shape)
else:
    # Original spectrum-based path for fine-tuned Lightning checkpoints.
    prec_ood = None
    prec_val = None

    if USE_CACHED_PREDICTIONS:
        with h5py.File(FINETUNING_HDF5_PATH, 'r') as f:
            fold = np.array([x.decode('utf-8') if isinstance(x, bytes) else str(x) for x in f['fold'][:]])
            smiles_all = np.array([x.decode('utf-8') if isinstance(x, bytes) else str(x) for x in f['smiles'][:]])

        val_mask = fold == 'val'
        if val_mask.sum() == 0:
            val_mask = fold == 'test'

        smiles_val = smiles_all[val_mask].tolist()
        spec_ood = None
        spec_val = None
        print('Using cached predictions; skipped spectrum tensor construction.')
        print('OOD smiles:', len(smiles_ood))
        print('VAL smiles:', len(smiles_val))
    else:
        spec_ood = np.stack([parse_spectrum_strings(m, i) for m, i in zip(df_ood['mzs'], df_ood['intensities'])])

        # OOD precursor mz from probing_test when available.
        for prec_col in ['precursor_mz', 'PRECURSOR_MZ', 'prec_mz', 'PRECURSOR M/Z']:
            if prec_col in df_ood.columns:
                prec_ood = pd.to_numeric(df_ood[prec_col], errors='coerce').to_numpy(dtype=np.float32)
                break

        with h5py.File(FINETUNING_HDF5_PATH, 'r') as f:
            fold = np.array([x.decode('utf-8') if isinstance(x, bytes) else str(x) for x in f['fold'][:]])
            smiles_all = np.array([x.decode('utf-8') if isinstance(x, bytes) else str(x) for x in f['smiles'][:]])
            spec_all = f['spectrum'][:]
            if 'precursor_mz' in f:
                prec_all = f['precursor_mz'][:].astype(np.float32)
            else:
                prec_all = None

        val_mask = fold == 'val'
        if val_mask.sum() == 0:
            val_mask = fold == 'test'

        smiles_val = smiles_all[val_mask].tolist()
        spec_val = spec_all[val_mask].astype(np.float32)
        if prec_all is not None:
            prec_val = prec_all[val_mask]

        print('OOD spectra:', len(smiles_ood), spec_ood.shape)
        if prec_ood is not None:
            print('OOD precursor_mz:', prec_ood.shape)
        print('VAL spectra:', len(smiles_val), spec_val.shape)
        if prec_val is not None:
            print('VAL precursor_mz:', prec_val.shape)

Using cached predictions; skipped spectrum tensor construction.
OOD smiles: 45185
VAL smiles: 23415


In [53]:
# ------------------------------------------------------------------
# Inference + ground truth fingerprints
# ------------------------------------------------------------------
if USE_CACHED_PREDICTIONS:
    y_pred_ood = np.load(CACHE_Y_PRED_PATH)
    y_true_ood = np.load(CACHE_Y_TRUE_PATH)
    y_pred_val = np.load(CACHE_Y_PRED_VAL_PATH)
    y_true_val = np.load(CACHE_Y_TRUE_VAL_PATH)
    print('Loaded cached predictions from', MODEL_OUTPUT_DIR)
else:
    if MODEL_TYPE == 'frozen_head':
        y_pred_ood = infer_frozen_batches(model, emb_ood, embedding_masks_np=emb_mask_ood)
        y_pred_val = infer_frozen_batches(model, emb_val, embedding_masks_np=emb_mask_val)
    else:
        y_pred_ood = infer_batches(model, spec_ood, prec_mz_np=prec_ood)
        y_pred_val = infer_batches(model, spec_val, prec_mz_np=prec_val)

    y_true_ood = np.stack([fp_from_smiles(s, FP_KIND) for s in tqdm(smiles_ood, desc='GT OOD FPs')]).astype(np.float32)
    y_true_val = np.stack([fp_from_smiles(s, FP_KIND) for s in tqdm(smiles_val, desc='GT VAL FPs')]).astype(np.float32)

if y_pred_ood.shape[1] != y_true_ood.shape[1]:
    raise ValueError(f'OOD dimension mismatch: pred={y_pred_ood.shape}, true={y_true_ood.shape}')
if y_pred_val.shape[1] != y_true_val.shape[1]:
    raise ValueError(f'VAL dimension mismatch: pred={y_pred_val.shape}, true={y_true_val.shape}')

if not USE_CACHED_PREDICTIONS:
    np.save(CACHE_Y_PRED_PATH, y_pred_ood)
    np.save(CACHE_Y_TRUE_PATH, y_true_ood)
    np.save(CACHE_Y_PRED_VAL_PATH, y_pred_val)
    np.save(CACHE_Y_TRUE_VAL_PATH, y_true_val)
    np.save(MODEL_OUTPUT_DIR / 'smiles.npy', np.array(smiles_ood, dtype=object))
    print('Saved prediction caches to', MODEL_OUTPUT_DIR)
print('Pred OOD shape:', y_pred_ood.shape)
print('Pred VAL shape:', y_pred_val.shape)

Loaded cached predictions from /Users/wouterachterberg/coding/DreaMS/dreams-thesis-wa/results/model_runs/map4_2048_bce/axis2_artifacts
Pred OOD shape: (45185, 2048)
Pred VAL shape: (23415, 2048)


In [54]:
# ------------------------------------------------------------------
# Part 1: per-bit AUROC val vs OOD
# ------------------------------------------------------------------
au_val = compute_per_bit_auroc(y_true_val, y_pred_val).rename(columns={'auroc': 'auroc_val', 'freq': 'freq_val'})
au_ood = compute_per_bit_auroc(y_true_ood, y_pred_ood).rename(columns={'auroc': 'auroc_ood', 'freq': 'freq_ood'})
au = au_val.merge(au_ood, on='bit_index', how='inner')
au['auroc_drop'] = au['auroc_val'] - au['auroc_ood']
au.to_csv(AUROC_DIR / 'auroc_comparison.csv', index=False)

worst = au.sort_values('auroc_drop', ascending=False).head(20)
best = au.sort_values('auroc_drop', ascending=True).head(20)
worst.to_csv(AUROC_DIR / 'worst_generalizing_bits.csv', index=False)
best.to_csv(AUROC_DIR / 'best_generalizing_bits.csv', index=False)

mean_per_bit_auroc_val = float(np.nanmean(au['auroc_val']))
mean_per_bit_auroc_ood = float(np.nanmean(au['auroc_ood']))
mean_per_bit_auroc_drop = float(np.nanmean(au['auroc_drop']))
summary = pd.DataFrame([
    {'metric': 'n_bits', 'value': int(len(au))},
    {'metric': 'mean_per_bit_auroc_val', 'value': mean_per_bit_auroc_val},
    {'metric': 'mean_per_bit_auroc_ood', 'value': mean_per_bit_auroc_ood},
    {'metric': 'mean_per_bit_auroc_drop', 'value': mean_per_bit_auroc_drop},
    {'metric': 'mean_auroc_val', 'value': mean_per_bit_auroc_val},
    {'metric': 'mean_auroc_ood', 'value': mean_per_bit_auroc_ood},
    {'metric': 'mean_auroc_drop', 'value': mean_per_bit_auroc_drop},
])
summary.to_csv(AUROC_DIR / 'auroc_summary_table.csv', index=False)

print('Saved per-bit AUROC artifacts to', AUROC_DIR)

Saved per-bit AUROC artifacts to /Users/wouterachterberg/coding/DreaMS/dreams-thesis-wa/results/model_runs/map4_2048_bce/axis2_artifacts/per_bit_auroc


In [55]:
# ------------------------------------------------------------------
# Part 2: threshold sweep
# ------------------------------------------------------------------
thresholds = np.round(np.linspace(0.01, 0.99, 99), 2)
sweep_val = sweep_thresholds(y_pred_val, y_true_val, thresholds)
sweep_ood = sweep_thresholds(y_pred_ood, y_true_ood, thresholds)
sweep_val.to_csv(SWEEP_DIR / 'sweep_results_val.csv', index=False)
sweep_ood.to_csv(SWEEP_DIR / 'sweep_results_ood.csv', index=False)

best_val = sweep_val.loc[sweep_val['tanimoto_mean'].idxmax()]
best_ood = sweep_ood.loc[sweep_ood['tanimoto_mean'].idxmax()]
comp = pd.DataFrame([
    {'split': 'val', 'best_tau_tanimoto': float(best_val['threshold']), 'best_tanimoto': float(best_val['tanimoto_mean'])},
    {'split': 'ood', 'best_tau_tanimoto': float(best_ood['threshold']), 'best_tanimoto': float(best_ood['tanimoto_mean'])},
])
comp.to_csv(SWEEP_DIR / 'threshold_comparison_table.csv', index=False)

print('Saved threshold sweep artifacts to', SWEEP_DIR)

/var/folders/21/cp9j844d6q74kpp18myp0wxr0000gn/T/ipykernel_93405/3397636381.py:214: RuntimeWarning: invalid value encountered in divide
  precision = np.where(tp + fp > 0, tp / (tp + fp), 0.0).mean()


Saved threshold sweep artifacts to /Users/wouterachterberg/coding/DreaMS/dreams-thesis-wa/results/model_runs/map4_2048_bce/axis2_artifacts/threshold_sweep


In [56]:
# ------------------------------------------------------------------
# Part 3: retrieval metrics
# ------------------------------------------------------------------
def build_library(smiles, fp_kind):
    uniq = list(dict.fromkeys(smiles))
    smi_to_idx = {s: i for i, s in enumerate(uniq)}
    lib = np.stack([fp_from_smiles(s, fp_kind) for s in uniq]).astype(np.float32)
    spec_to_mol = np.array([smi_to_idx[s] for s in smiles], dtype=np.int32)
    return lib, spec_to_mol

def compute_ranks(y_pred, lib, spec_to_mol):
    sims = cosine_similarity(y_pred, lib)
    ranks = np.zeros((len(y_pred),), dtype=np.int32)
    for i in range(len(y_pred)):
        s = sims[i, spec_to_mol[i]]
        ranks[i] = int((sims[i] > s).sum() + 1)
    return ranks

def retrieval_metrics(ranks, library_size):
    out = {
        'library_size': library_size,
        'n_spectra': int(len(ranks)),
        'acc@1': float((ranks <= 1).mean()),
        'acc@5': float((ranks <= 5).mean()),
        'acc@10': float((ranks <= 10).mean()),
        'acc@20': float((ranks <= 20).mean()),
        'acc@50': float((ranks <= 50).mean()),
        'acc@100': float((ranks <= 100).mean()),
        'mrr': float((1.0 / ranks).mean()),
        'mean_rank': float(ranks.mean()),
        'median_rank': float(np.median(ranks)),
    }
    return out

cos_val = compute_rowwise_cosine_stats(y_true_val, y_pred_val)
cos_ood = compute_rowwise_cosine_stats(y_true_ood, y_pred_ood)
cosine_metrics_df = pd.DataFrame([
    {'Metric': 'cosine_sim_mean', 'Validation': cos_val['cosine_sim_mean'], 'OOD': cos_ood['cosine_sim_mean']},
    {'Metric': 'cosine_sim_median', 'Validation': cos_val['cosine_sim_median'], 'OOD': cos_ood['cosine_sim_median']},
])
cosine_metrics_df.to_csv(RETRIEVAL_DIR / 'cosine_similarity_metrics.csv', index=False)

lib_val, map_val = build_library(smiles_val, FP_KIND)
lib_ood, map_ood = build_library(smiles_ood, FP_KIND)
ranks_val = compute_ranks(y_pred_val, lib_val, map_val)
ranks_ood = compute_ranks(y_pred_ood, lib_ood, map_ood)
np.save(RETRIEVAL_DIR / 'ranks_val.npy', ranks_val)
np.save(RETRIEVAL_DIR / 'ranks_ood.npy', ranks_ood)

m_val = retrieval_metrics(ranks_val, len(lib_val))
m_ood = retrieval_metrics(ranks_ood, len(lib_ood))
metrics_rows = [
    {'Metric': 'cosine_sim_mean', 'Validation': cos_val['cosine_sim_mean'], 'OOD': cos_ood['cosine_sim_mean']},
    {'Metric': 'cosine_sim_median', 'Validation': cos_val['cosine_sim_median'], 'OOD': cos_ood['cosine_sim_median']},
]
metrics_rows.extend([
    {'Metric': k, 'Validation': m_val[k], 'OOD': m_ood[k]}
    for k in ['library_size', 'n_spectra', 'acc@1', 'acc@5', 'acc@10', 'acc@20', 'acc@50', 'acc@100', 'mrr', 'mean_rank', 'median_rank']
])
metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(RETRIEVAL_DIR / 'retrieval_metrics.csv', index=False)
metrics_df.to_csv(RETRIEVAL_DIR / 'retrieval_comparison_table.csv', index=False)

print('Saved retrieval artifacts to', RETRIEVAL_DIR)

Saved retrieval artifacts to /Users/wouterachterberg/coding/DreaMS/dreams-thesis-wa/results/model_runs/map4_2048_bce/axis2_artifacts/retrieval


In [57]:
# Manifest and run metadata
manifest = sorted([str(p.relative_to(MODEL_RUN_DIR)) for p in MODEL_RUN_DIR.rglob('*') if p.is_file()])
pd.DataFrame({'file': manifest}).to_csv(MODEL_OUTPUT_DIR / 'artifact_manifest.csv', index=False)

meta = {
    'run_tag': RUN_TAG,
    'checkpoint': str(CKPT_PATH),
    'fp_kind': FP_KIND,
    'loss_kind': LOSS_KIND,
    'apply_sigmoid_to_pred': bool(APPLY_SIGMOID_TO_PRED),
    'model_run_dir': str(MODEL_RUN_DIR),
    'artifacts_output_dir': str(MODEL_OUTPUT_DIR),
}
(MODEL_OUTPUT_DIR / 'run_config.json').write_text(json.dumps(meta, indent=2))

print('Done. Artifacts written to:', MODEL_OUTPUT_DIR)
print('Checkpoint stored at:', CKPT_PATH)
display(pd.DataFrame({'file': manifest}).head(40))

Done. Artifacts written to: /Users/wouterachterberg/coding/DreaMS/dreams-thesis-wa/results/model_runs/map4_2048_bce/axis2_artifacts
Checkpoint stored at: /Users/wouterachterberg/coding/DreaMS/dreams-thesis-wa/results/model_runs/map4_2048_bce/checkpoints/epoch=17-step=2310-val_loss=0.453639.ckpt


,file
0,axis2_artifacts/inference_only_metadata.json
1,axis2_artifacts/per_bit_auroc/auroc_comparison...
2,axis2_artifacts/per_bit_auroc/auroc_summary_ta...
3,axis2_artifacts/per_bit_auroc/best_generalizin...
4,axis2_artifacts/per_bit_auroc/worst_generalizi...
5,axis2_artifacts/retrieval/ranks_ood.npy
6,axis2_artifacts/retrieval/ranks_val.npy
7,axis2_artifacts/retrieval/retrieval_comparison...
8,axis2_artifacts/retrieval/retrieval_metrics.csv
9,axis2_artifacts/threshold_sweep/sweep_results_...
